In [8]:
import requests
import pandas as pd
import numpy as np
import cvxpy as cp
import ecos as ecos
import os
from tqdm import tqdm

In [2]:
FMP_API_KEY = "ksnxCfuMd8YcYtNVDrqZKBY0aZeDMLX8"

In [10]:
def get_stock_list(csv_filename="stock_list.csv"):
    """
    Fetches all stocks and indices with a market cap > $100M from FMP,
    using a local CSV if available. Returns a list of tickers.
    """
    if os.path.exists(csv_filename):
        # If the CSV file exists, read from it
        df = pd.read_csv(csv_filename)
        tickers = df["symbol"].tolist()
        print(f"Loaded {len(tickers)} tickers from local CSV: {csv_filename}")
        return tickers
    else:
        # If the CSV file doesn't exist, call the API and save
        url = f"https://financialmodelingprep.com/api/v3/stock-screener?marketCapMoreThan=100000000&apikey={FMP_API_KEY}"
        response = requests.get(url)
        data = response.json()
        
        # We can add a progress bar for this loop (though typically it's not very large):
        tickers = []
        for stock in tqdm(data, desc="Fetching ticker symbols"):
            if "symbol" in stock:
                tickers.append(stock["symbol"])
        
        # Save the data to CSV
        df = pd.DataFrame(tickers, columns=["symbol"])
        df.to_csv(csv_filename, index=False)
        print(f"Saved {len(tickers)} tickers to local CSV: {csv_filename}")

        return tickers

In [11]:
def get_historical_prices(tickers, period="10y", data_dir="historical_data"):
    """
    Fetches historical price data for given tickers.
    Saves each ticker's historical data as a CSV in 'data_dir'.
    Returns a DataFrame with dates as index and tickers as columns, sorted in ascending order.
    """
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
    
    price_data = {}
    
    # Wrap the ticker loop in a tqdm progress bar
    for ticker in tqdm(tickers, desc="Downloading/loading historical prices"):
        csv_path = os.path.join(data_dir, f"{ticker}.csv")

        if os.path.exists(csv_path):
            # Load from local CSV
            df = pd.read_csv(csv_path, parse_dates=["date"], index_col="date")
            # Print or log progress if desired
        else:
            # Download from FMP
            url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{ticker}?serietype=line&apikey={FMP_API_KEY}"
            response = requests.get(url)
            data = response.json()

            if "historical" not in data:
                # Log progress if no data
                continue

            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df.set_index("date", inplace=True)

            # Save to CSV
            df.to_csv(csv_path)
        
        if not df.empty:
            price_data[ticker] = df["close"]
    
    # Create a DataFrame and ensure it is sorted by date in ascending order
    df_prices = pd.DataFrame(price_data).sort_index(ascending=True)
    
    return df_prices

In [9]:
def compute_metrics(price_df):
    """
    Computes 1Y CAGR, 5Y CAGR, 1Y SD, and 5Y SD for each stock.
    Returns a DataFrame with tickers as index and metrics as columns.
    """
    metrics = {}
    
    # Wrap iteration over columns in tqdm
    for ticker in tqdm(price_df.columns, desc="Computing metrics"):
        prices = price_df[ticker].dropna()  # keep non-NaNs
        if len(prices) < 252:  # Not enough data for 1 year
            continue

        # Most recent date
        end_1y = prices.index[-1]
        start_1y = end_1y - pd.DateOffset(years=1)
        start_1y = prices.index.asof(start_1y)

        cagr_1y = (prices.loc[end_1y] / prices.loc[start_1y]) ** (1/1) - 1
        sd_1y = np.std(prices.pct_change().dropna()) * np.sqrt(252)

        # For 5Y
        if len(prices) >= 252*5:
            end_5y = prices.index[-1]
            start_5y = end_5y - pd.DateOffset(years=5)
            start_5y = prices.index.asof(start_5y)
            cagr_5y = (prices.loc[end_5y] / prices.loc[start_5y]) ** (1/5) - 1
            sd_5y = np.std(prices.pct_change().dropna()) * np.sqrt(252)
        else:
            cagr_5y, sd_5y = np.nan, np.nan

        metrics[ticker] = [cagr_1y, cagr_5y, sd_1y, sd_5y]

    # Build and return DataFrame
    return pd.DataFrame.from_dict(metrics, orient="index", 
                                  columns=["1Y CAGR", "5Y CAGR", "1Y SD", "5Y SD"])

In [30]:
def optimize_portfolio(metrics_df, target_return=0.1, use_5y=True, 
                       allow_short=False, max_weight=0.3, num_stocks=10):
    """
    Optimizes portfolio allocation using Mean-Variance Optimization (MVO).
    Shows final weights as decimal fractions (e.g., 0.12 = 12%).
    """
    # Choose columns
    expected_returns = metrics_df["5Y CAGR"] if use_5y else metrics_df["1Y CAGR"]
    risk_matrix = metrics_df["5Y SD"] if use_5y else metrics_df["1Y SD"]

    # Drop NaN
    valid_stocks = expected_returns.dropna().index.intersection(risk_matrix.dropna().index)
    expected_returns = expected_returns.loc[valid_stocks]
    risk_matrix = risk_matrix.loc[valid_stocks]

    # Rank by risk-adjusted return
    risk_adjusted_return = expected_returns / risk_matrix
    candidate_stocks = risk_adjusted_return.nlargest(num_stocks * 2).index

    # Fetch historical prices for only selected stocks
    returns_df = get_historical_prices(candidate_stocks)
    
    # Compute log returns and covariance
    log_returns = np.log(returns_df / returns_df.shift(1)).dropna()
    cov_matrix = log_returns.cov()

    # Ensure shapes match
    selected_stocks = expected_returns.loc[candidate_stocks].index.intersection(cov_matrix.index)
    expected_returns = expected_returns.loc[selected_stocks]
    cov_matrix = cov_matrix.loc[selected_stocks, selected_stocks]

    n = len(selected_stocks)
    if n == 0:
        return "No feasible stocks found."

    # Create cvxpy variables
    weights = cp.Variable(n)
    z = cp.Variable(n, boolean=True)

    # Objective: minimize variance
    portfolio_variance = cp.quad_form(weights, cov_matrix)
    objective = cp.Minimize(portfolio_variance)

    # Constraints
    constraints = [
        cp.sum(weights) == 1,
        expected_returns.to_numpy() @ weights >= target_return
    ]
    
    if not allow_short:
        constraints.append(weights >= 0)
    constraints.append(weights <= max_weight * z)
    constraints.append(cp.sum(z) <= num_stocks)

    # Solve
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.MOSEK if cp.MOSEK in cp.installed_solvers() else cp.ECOS_BB)

    # Get results
    if prob.status not in ["optimal", "optimal_inaccurate"]:
        return "No feasible solution found."

    optimal_weights = weights.value

    # Filter out negligible weights <= 1e-4 (optional)
    #selected_stocks = selected_stocks[optimal_weights > 1e-4]

    # Build a dict of ticker -> allocation in decimal form (e.g., 0.12 means 12%)
    final_weights = {
        ticker: float(w)
        for ticker, w in zip(selected_stocks, optimal_weights)
        if w > 1e-4
    }

    # Compute portfolio stats
    expected_portfolio_return = expected_returns.to_numpy() @ optimal_weights
    daily_portfolio_variance = optimal_weights.T @ cov_matrix.to_numpy() @ optimal_weights
    annualized_portfolio_variance = daily_portfolio_variance * 252
    annualized_portfolio_std = np.sqrt(annualized_portfolio_variance)

    return {
        "tickers": list(selected_stocks),
        "weights": final_weights,  # decimal fractions
        "expected_return": float(round(expected_portfolio_return * 100, 2)),  # still in %
        "portfolio_std": float(round(annualized_portfolio_std * 100, 2))      # still in %
    }


In [13]:
list_of_tickers = get_stock_list()

Fetching ticker symbols: 100%|██████████| 1000/1000 [00:00<?, ?it/s]

Saved 1000 tickers to local CSV: stock_list.csv


In [14]:
price_df = get_historical_prices(tickers = list_of_tickers)

Downloading/loading historical prices: 100%|██████████| 1000/1000 [10:51<00:00,  1.54it/s]


In [15]:
metrics = compute_metrics(price_df)

Computing metrics: 100%|██████████| 999/999 [00:02<00:00, 466.00it/s]


In [33]:
optimize_portfolio(metrics, target_return=0.5, use_5y=True, allow_short=False, max_weight=0.3, num_stocks=10)

Downloading/loading historical prices: 100%|██████████| 20/20 [00:00<00:00, 74.04it/s]


{'tickers': ['TPL',
  'LLY',
  'TRGP',
  'AVGO',
  'GE',
  'IMO.TO',
  'NVDA',
  'DOL.TO',
  'EQT',
  'IMO',
  'VST',
  'PANW',
  'MSTR',
  'MCK',
  'CSU.TO',
  'AJG',
  'HWM',
  'RMS.PA',
  'ANET',
  'L.TO'],
 'weights': {'TPL': 0.04306501059380065,
  'LLY': 0.2348766168848874,
  'TRGP': 0.08454573505996357,
  'NVDA': 0.13138060313102198,
  'DOL.TO': 0.13363859581366147,
  'VST': 0.04442507625354602,
  'MSTR': 0.06489088518665212,
  'MCK': 0.11050847487988344,
  'RMS.PA': 0.15266900219513482},
 'expected_return': 50.0,
 'portfolio_std': 20.53}